# U-Net vs SegFormer-B2 Comparison on Crack500

This notebook compares the saved `U-Net` baseline and the saved `SegFormer-B2` baseline on the same `Crack500` test samples.

Use it to:
- compare the same fixed samples side by side
- check full-test metrics for both models in one place
- inspect which samples improve most or regress most under SegFormer-B2


In [ ]:
import os
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from dataset import CrackDataset
from loss import BCEDiceLoss
from metrics import f1_score, iou_score
from model import get_model

ROOT = "../CRACK500"
img_size = 360
batch_size = 8
threshold = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sample_indices = [0, 5, 12, 25, 40]
print(f"Using device: {device}")
print(f"Fixed sample indices: {sample_indices}")



In [ ]:
MODEL_CONFIGS = {
    "U-Net": {
        "model_name": "Unet",
        "encoder_name": "resnet34",
        "encoder_weights": "imagenet",
        "checkpoint_path": os.path.join("..", "checkpoints", "best_model.pth"),
    },
    "SegFormer-B2": {
        "model_name": "SegFormer-B2",
        "encoder_name": "resnet34",
        "encoder_weights": "imagenet",
        "checkpoint_path": os.path.join("..", "checkpoints", "segformer_b2_best.pth"),
    },
}

plt.rcParams["figure.dpi"] = 120

for model_label, cfg in MODEL_CONFIGS.items():
    print(f"{model_label:13s} -> {cfg['checkpoint_path']}")


## 1. Load the Test Dataset and Both Models

We keep the dataset and preprocessing identical across both models so the visual comparison stays fair.

In [ ]:
test_dataset = CrackDataset(ROOT, split="test", img_size=img_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
criterion = BCEDiceLoss()


def load_model_from_config(cfg):
    checkpoint_path = cfg["checkpoint_path"]
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    model = get_model(
        model_name=cfg["model_name"],
        encoder_name=cfg["encoder_name"],
        encoder_weights=cfg["encoder_weights"],
        in_channels=3,
        classes=1,
    )
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model


models = {label: load_model_from_config(cfg) for label, cfg in MODEL_CONFIGS.items()}

print(f"Loaded test dataset with {len(test_dataset)} samples.")
print("Loaded models:", ", ".join(models.keys()))


## 2. Helper Functions

Each helper runs inference on one sample, computes per-sample metrics, and resizes the predicted mask back to the original image size for plotting.

In [ ]:
def make_overlay(image_np, mask_np, color=(255, 0, 0), alpha=0.45):
    overlay = image_np.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    overlay[mask_np == 1] = overlay[mask_np == 1] * (1 - alpha) + color_arr * alpha
    return overlay.clip(0, 255).astype(np.uint8)


def predict_one(model, index, threshold=threshold):
    raw_image, raw_mask = test_dataset.get_raw(index)

    image_t, mask_t = test_dataset[index]
    image_t = image_t.unsqueeze(0).to(device)
    mask_t = mask_t.unsqueeze(0).unsqueeze(1).float().to(device)

    with torch.no_grad():
        logits = model(image_t)
        probs = torch.sigmoid(logits)
        pred_mask_t = (probs > threshold).float()
        loss = criterion(logits, mask_t).item()
        iou = iou_score(logits, mask_t, threshold=threshold)
        f1 = f1_score(logits, mask_t, threshold=threshold)

    raw_h, raw_w = raw_mask.shape
    pred_mask = F.interpolate(
        pred_mask_t,
        size=(raw_h, raw_w),
        mode="nearest",
    )[0, 0].cpu().numpy().astype(np.uint8)

    return {
        "image": raw_image,
        "gt_mask": raw_mask,
        "pred_mask": pred_mask,
        "pred_overlay": make_overlay(raw_image, pred_mask),
        "loss": loss,
        "iou": iou,
        "f1": f1,
    }


def evaluate_model(model):
    total_loss = 0.0
    total_iou = 0.0
    total_f1 = 0.0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.unsqueeze(1).float().to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            batch_size_now = images.size(0)
            total_loss += loss.item() * batch_size_now
            total_iou += iou_score(outputs, masks, threshold=threshold) * batch_size_now
            total_f1 += f1_score(outputs, masks, threshold=threshold) * batch_size_now

    dataset_size = len(test_dataset)
    return {
        "loss": total_loss / dataset_size,
        "iou": total_iou / dataset_size,
        "f1": total_f1 / dataset_size,
    }


## 3. Full-Test Metric Comparison

This gives a quick quantitative summary before looking at individual qualitative examples.

In [ ]:
metrics_by_model = {label: evaluate_model(model) for label, model in models.items()}

print("Test-set summary")
print("-" * 58)
print(f"{'Model':13s} | {'Loss':>8s} | {'IoU':>8s} | {'F1':>8s}")
print("-" * 58)
for label, stats in metrics_by_model.items():
    print(f"{label:13s} | {stats['loss']:8.4f} | {stats['iou']:8.4f} | {stats['f1']:8.4f}")

delta_iou = metrics_by_model['SegFormer-B2']['iou'] - metrics_by_model['U-Net']['iou']
delta_f1 = metrics_by_model['SegFormer-B2']['f1'] - metrics_by_model['U-Net']['f1']

print("-" * 58)
print(f"SegFormer-B2 minus U-Net | IoU delta={delta_iou:+.4f} | F1 delta={delta_f1:+.4f}")


## 4. Fixed-Sample Side-by-Side Visualization

These are the same fixed indices used in the earlier notebooks, so the comparison is directly aligned.

In [ ]:
num_rows = len(sample_indices)
fig, axes = plt.subplots(num_rows, 6, figsize=(18, 3.2 * num_rows))

if num_rows == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = [
    'Image',
    'Ground Truth',
    'U-Net Pred',
    'U-Net Overlay',
    'SegFormer Pred',
    'SegFormer Overlay',
]
for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, idx in enumerate(sample_indices):
    unet_sample = predict_one(models['U-Net'], idx)
    segformer_sample = predict_one(models['SegFormer-B2'], idx)

    axes[row, 0].imshow(unet_sample['image'])
    axes[row, 1].imshow(unet_sample['gt_mask'], cmap='gray', vmin=0, vmax=1)
    axes[row, 2].imshow(unet_sample['pred_mask'], cmap='gray', vmin=0, vmax=1)
    axes[row, 3].imshow(unet_sample['pred_overlay'])
    axes[row, 4].imshow(segformer_sample['pred_mask'], cmap='gray', vmin=0, vmax=1)
    axes[row, 5].imshow(segformer_sample['pred_overlay'])

    axes[row, 0].set_ylabel(
        f"idx={idx}\n"
        f"U IoU={unet_sample['iou']:.3f}\n"
        f"S IoU={segformer_sample['iou']:.3f}",
        rotation=0,
        labelpad=40,
        va='center',
        fontsize=9,
    )

    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
plt.show()


## 5. Rank Samples by Improvement or Regression

This cell helps you find where `SegFormer-B2` helps most and where it falls behind `U-Net`.

In [ ]:
per_sample_rows = []

for idx in range(len(test_dataset)):
    unet_sample = predict_one(models['U-Net'], idx)
    segformer_sample = predict_one(models['SegFormer-B2'], idx)
    per_sample_rows.append({
        'idx': idx,
        'unet_iou': unet_sample['iou'],
        'segformer_iou': segformer_sample['iou'],
        'iou_delta': segformer_sample['iou'] - unet_sample['iou'],
        'unet_f1': unet_sample['f1'],
        'segformer_f1': segformer_sample['f1'],
        'f1_delta': segformer_sample['f1'] - unet_sample['f1'],
    })

top_gains = sorted(per_sample_rows, key=lambda x: x['iou_delta'], reverse=True)[:10]
top_regressions = sorted(per_sample_rows, key=lambda x: x['iou_delta'])[:10]

print('Top 10 SegFormer-B2 gains over U-Net (by IoU delta)')
for row in top_gains:
    print(
        f"idx={row['idx']:4d} | "
        f"U-Net IoU={row['unet_iou']:.4f} | "
        f"SegFormer IoU={row['segformer_iou']:.4f} | "
        f"delta={row['iou_delta']:+.4f}"
    )

print()
print('Top 10 SegFormer-B2 regressions vs U-Net (by IoU delta)')
for row in top_regressions:
    print(
        f"idx={row['idx']:4d} | "
        f"U-Net IoU={row['unet_iou']:.4f} | "
        f"SegFormer IoU={row['segformer_iou']:.4f} | "
        f"delta={row['iou_delta']:+.4f}"
    )


## 6. Deep Dive Into U-Net Worst Cases

These samples come from the largest positive `IoU` deltas in the ranking above. The goal is to inspect **how** `SegFormer-B2` behaves on the same hard samples and **why** it wins over `U-Net` in those cases.

Legend for the error map: `green = true positive`, `red = false positive`, `yellow = false negative`.

A useful sanity check here is the threshold sweep. If `U-Net` is still poor even after scanning thresholds from `0.05` to `0.95`, then the failure is not just bad calibration at `0.5`; it is a real representation/localization failure.

In [ ]:
def confusion_counts(pred_mask, gt_mask):
    tp = int(((pred_mask == 1) & (gt_mask == 1)).sum())
    fp = int(((pred_mask == 1) & (gt_mask == 0)).sum())
    fn = int(((pred_mask == 0) & (gt_mask == 1)).sum())
    tn = int(((pred_mask == 0) & (gt_mask == 0)).sum())
    return tp, fp, fn, tn


def binary_stats(pred_mask, gt_mask):
    tp, fp, fn, tn = confusion_counts(pred_mask, gt_mask)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-6)
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "iou": iou,
        "f1": f1,
        "pred_pos": int(pred_mask.sum()),
        "gt_pos": int(gt_mask.sum()),
    }


def error_map(pred_mask, gt_mask):
    error_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    error_rgb[(pred_mask == 1) & (gt_mask == 1)] = [0, 200, 0]
    error_rgb[(pred_mask == 1) & (gt_mask == 0)] = [220, 0, 0]
    error_rgb[(pred_mask == 0) & (gt_mask == 1)] = [255, 215, 0]
    return error_rgb


def best_threshold_stats(prob_map, gt_mask, thresholds=np.linspace(0.05, 0.95, 19)):
    best = None
    for t in thresholds:
        pred_mask = (prob_map > t).astype(np.uint8)
        stats = binary_stats(pred_mask, gt_mask)
        if best is None or stats["iou"] > best["iou"]:
            best = {"threshold": float(t), **stats}
    return best


def predict_detail(model, index, threshold=threshold):
    raw_image, raw_mask = test_dataset.get_raw(index)
    image_t, _ = test_dataset[index]
    image_t = image_t.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image_t)
        probs = torch.sigmoid(logits)

    raw_h, raw_w = raw_mask.shape
    prob_map = F.interpolate(
        probs,
        size=(raw_h, raw_w),
        mode="bilinear",
        align_corners=False,
    )[0, 0].cpu().numpy()

    pred_mask = (prob_map > threshold).astype(np.uint8)
    gt_mask = raw_mask.astype(np.uint8)
    stats = binary_stats(pred_mask, gt_mask)
    best_stats = best_threshold_stats(prob_map, gt_mask)
    gt_probs = prob_map[gt_mask == 1]
    bg_probs = prob_map[gt_mask == 0]

    return {
        "image": raw_image,
        "gt_mask": gt_mask,
        "prob_map": prob_map,
        "pred_mask": pred_mask,
        "pred_overlay": make_overlay(raw_image, pred_mask),
        "error_map": error_map(pred_mask, gt_mask),
        "stats": stats,
        "best_stats": best_stats,
        "mean_prob_on_gt": float(gt_probs.mean()) if gt_probs.size else 0.0,
        "p90_prob_on_gt": float(np.quantile(gt_probs, 0.9)) if gt_probs.size else 0.0,
        "max_prob_on_gt": float(gt_probs.max()) if gt_probs.size else 0.0,
        "mean_prob_on_bg": float(bg_probs.mean()) if bg_probs.size else 0.0,
        "p99_prob_on_bg": float(np.quantile(bg_probs, 0.99)) if bg_probs.size else 0.0,
    }

In [ ]:
focus_indices = [420, 278]
print(f"Focus indices: {focus_indices}")

column_titles = [
    "Image",
    "Ground Truth",
    "U-Net Prob",
    "U-Net Pred",
    "U-Net Error",
    "SegFormer Prob",
    "SegFormer Pred",
    "SegFormer Error",
]

fig, axes = plt.subplots(len(focus_indices), len(column_titles), figsize=(24, 4.5 * len(focus_indices)))
if len(focus_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, idx in enumerate(focus_indices):
    unet_detail = predict_detail(models["U-Net"], idx)
    segformer_detail = predict_detail(models["SegFormer-B2"], idx)

    axes[row, 0].imshow(unet_detail["image"])
    axes[row, 1].imshow(unet_detail["gt_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 2].imshow(unet_detail["prob_map"], cmap="magma", vmin=0, vmax=1)
    axes[row, 3].imshow(unet_detail["pred_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 4].imshow(unet_detail["error_map"])
    axes[row, 5].imshow(segformer_detail["prob_map"], cmap="magma", vmin=0, vmax=1)
    axes[row, 6].imshow(segformer_detail["pred_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 7].imshow(segformer_detail["error_map"])

    axes[row, 0].set_ylabel(
        f"idx={idx}\n"
        f"U IoU={unet_detail['stats']['iou']:.3f}\n"
        f"S IoU={segformer_detail['stats']['iou']:.3f}",
        rotation=0,
        labelpad=38,
        va="center",
        fontsize=9,
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

for idx in focus_indices:
    unet_detail = predict_detail(models["U-Net"], idx)
    segformer_detail = predict_detail(models["SegFormer-B2"], idx)
    gt_ratio = unet_detail["gt_mask"].mean()
    recall_gain = segformer_detail["stats"]["recall"] - unet_detail["stats"]["recall"]
    iou_gain = segformer_detail["stats"]["iou"] - unet_detail["stats"]["iou"]
    mean_gt_prob_gain = segformer_detail["mean_prob_on_gt"] - unet_detail["mean_prob_on_gt"]

    print(f"\nidx={idx} | GT positive ratio={gt_ratio:.4f}")
    for label, detail in [("U-Net", unet_detail), ("SegFormer-B2", segformer_detail)]:
        stats = detail["stats"]
        best = detail["best_stats"]
        print(
            f"{label:13s} | IoU@0.5={stats['iou']:.4f} | P={stats['precision']:.4f} | R={stats['recall']:.4f} | "
            f"TP={stats['tp']:5d} | FP={stats['fp']:5d} | FN={stats['fn']:5d} | "
            f"mean(GT prob)={detail['mean_prob_on_gt']:.4f} | max(GT prob)={detail['max_prob_on_gt']:.4f} | "
            f"best IoU sweep={best['iou']:.4f} @ t={best['threshold']:.2f}"
        )

    print(
        f"Interpretation: SegFormer gains IoU by {iou_gain:+.4f} and recall by {recall_gain:+.4f}; "
        f"its mean probability on crack pixels is {mean_gt_prob_gain:+.4f} higher."
    )
    if unet_detail["best_stats"]["iou"] < 0.10:
        print(
            f"  U-Net still tops out at IoU={unet_detail['best_stats']['iou']:.4f} even after threshold sweep, "
            "so this is not mainly a bad 0.5 threshold. It is a real miss of the crack structure."
        )
    if segformer_detail["stats"]["recall"] > 0.70:
        print(
            "  SegFormer keeps the crack much more continuous across weak or noisy sections, "
            "although it also introduces some extra false positives around the crack boundary."
        )
    if idx == 420:
        print(
            "  Sample 420 is dominated by coarse gravel texture. U-Net only fires on tiny hotspots, "
            "while SegFormer recovers the long diagonal crack as one connected structure."
        )
    if idx == 278:
        print(
            "  Sample 278 contains a long, curved crack with brightness variation and edge fragments. "
            "U-Net locks onto a tiny bright segment; SegFormer follows the full path much more reliably."
        )

## 7. Pattern Summary Across Top Gains and Regressions

The two failure cases above are useful, but they are still just two samples. This section checks whether the same pattern still appears when we compare the full `top_gains` and `top_regressions` groups.

The summary below focuses on a few simple descriptors:
- how sparse the crack mask is
- how long or frame-spanning the crack is
- how thin the crack is inside its bounding box
- how strong the local contrast is around the crack
- whether `SegFormer-B2` wins mostly by recall or by precision

In [ ]:
def dilate_mask(mask_np, kernel_size=11):
    mask_t = torch.from_numpy(mask_np.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    dilated = F.max_pool2d(mask_t, kernel_size=kernel_size, stride=1, padding=kernel_size // 2)
    return dilated[0, 0].numpy().astype(np.uint8)


def extract_case_features(index):
    image_np, gt_mask = test_dataset.get_raw(index)
    gt_mask = gt_mask.astype(np.uint8)
    gray = image_np.astype(np.float32).mean(axis=2) / 255.0
    fg = gt_mask == 1

    ys, xs = np.where(fg)
    h, w = gt_mask.shape
    image_diag = np.sqrt(h ** 2 + w ** 2)

    if len(xs) == 0:
        x_span = y_span = diag_span = bbox_area_ratio = bbox_fill_ratio = 0.0
        border_touch_count = 0
    else:
        x0, x1 = xs.min(), xs.max()
        y0, y1 = ys.min(), ys.max()
        box_w = x1 - x0 + 1
        box_h = y1 - y0 + 1
        x_span = box_w / w
        y_span = box_h / h
        diag_span = np.sqrt(box_w ** 2 + box_h ** 2) / image_diag
        bbox_area_ratio = (box_w * box_h) / (h * w)
        bbox_fill_ratio = fg.mean() / (bbox_area_ratio + 1e-6)
        border_touch_count = (
            int(fg[0, :].any())
            + int(fg[-1, :].any())
            + int(fg[:, 0].any())
            + int(fg[:, -1].any())
        )

    ring = (dilate_mask(gt_mask, kernel_size=11) == 1) & (~fg)
    fg_mean = float(gray[fg].mean()) if fg.any() else 0.0
    ring_mean = float(gray[ring].mean()) if ring.any() else 0.0
    ring_std = float(gray[ring].std()) if ring.any() else 0.0
    local_contrast = abs(fg_mean - ring_mean)

    unet_detail = predict_detail(models["U-Net"], index)
    segformer_detail = predict_detail(models["SegFormer-B2"], index)

    return {
        "idx": index,
        "fg_ratio": float(fg.mean()),
        "x_span_ratio": x_span,
        "y_span_ratio": y_span,
        "diag_span_ratio": diag_span,
        "bbox_area_ratio": bbox_area_ratio,
        "bbox_fill_ratio": bbox_fill_ratio,
        "border_touch_count": float(border_touch_count),
        "local_contrast": local_contrast,
        "ring_texture_std": ring_std,
        "unet_iou": unet_detail["stats"]["iou"],
        "segformer_iou": segformer_detail["stats"]["iou"],
        "iou_delta": segformer_detail["stats"]["iou"] - unet_detail["stats"]["iou"],
        "unet_recall": unet_detail["stats"]["recall"],
        "segformer_recall": segformer_detail["stats"]["recall"],
        "recall_delta": segformer_detail["stats"]["recall"] - unet_detail["stats"]["recall"],
        "unet_precision": unet_detail["stats"]["precision"],
        "segformer_precision": segformer_detail["stats"]["precision"],
        "precision_delta": segformer_detail["stats"]["precision"] - unet_detail["stats"]["precision"],
        "unet_best_iou": unet_detail["best_stats"]["iou"],
        "segformer_best_iou": segformer_detail["best_stats"]["iou"],
        "unet_best_threshold": unet_detail["best_stats"]["threshold"],
        "segformer_best_threshold": segformer_detail["best_stats"]["threshold"],
    }


def summarize_group(rows):
    keys = [
        "fg_ratio",
        "x_span_ratio",
        "y_span_ratio",
        "diag_span_ratio",
        "bbox_area_ratio",
        "bbox_fill_ratio",
        "border_touch_count",
        "local_contrast",
        "ring_texture_std",
        "iou_delta",
        "recall_delta",
        "precision_delta",
        "unet_best_iou",
        "segformer_best_iou",
    ]
    return {key: float(np.mean([row[key] for row in rows])) for key in keys}


gain_indices = [row["idx"] for row in top_gains]
regression_indices = [row["idx"] for row in top_regressions]

gain_cases = [extract_case_features(idx) for idx in gain_indices]
regression_cases = [extract_case_features(idx) for idx in regression_indices]

gain_summary = summarize_group(gain_cases)
regression_summary = summarize_group(regression_cases)

print("Top-gain indices:", gain_indices)
print("Top-regression indices:", regression_indices)
print()

print("Average descriptors by group")
print(
    f"{'Metric':20s} | {'Top Gains':>10s} | {'Top Regressions':>15s} | {'Gain-Reg Diff':>13s}"
)
print("-" * 70)
metric_order = [
    ("fg_ratio", "fg_ratio"),
    ("diag_span_ratio", "diag_span"),
    ("bbox_fill_ratio", "bbox_fill"),
    ("border_touch_count", "border_touch"),
    ("local_contrast", "local_contrast"),
    ("ring_texture_std", "ring_texture"),
    ("recall_delta", "recall_delta"),
    ("precision_delta", "precision_delta"),
    ("unet_best_iou", "unet_best_iou"),
]
for key, label in metric_order:
    gain_val = gain_summary[key]
    reg_val = regression_summary[key]
    diff = gain_val - reg_val
    print(f"{label:20s} | {gain_val:10.4f} | {reg_val:15.4f} | {diff:13.4f}")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
plot_metrics = [
    ("fg_ratio", "GT positive ratio"),
    ("diag_span_ratio", "Mask diagonal span"),
    ("bbox_fill_ratio", "BBox fill ratio"),
    ("border_touch_count", "Border touch count"),
    ("local_contrast", "Local crack contrast"),
    ("ring_texture_std", "Nearby texture std"),
    ("recall_delta", "SegFormer recall gain"),
    ("precision_delta", "SegFormer precision gain"),
]

for ax, (key, title) in zip(axes.ravel(), plot_metrics):
    ax.bar(
        ["Top Gains", "Top Regressions"],
        [gain_summary[key], regression_summary[key]],
        color=["#d95f02", "#1b9e77"],
    )
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print()
if gain_summary["bbox_fill_ratio"] < regression_summary["bbox_fill_ratio"]:
    print(
        "Pattern signal: the gain group has a lower bbox-fill ratio, so SegFormer helps more on thinner, more line-like cracks."
    )
if gain_summary["diag_span_ratio"] > regression_summary["diag_span_ratio"]:
    print(
        "Pattern signal: the gain group spans a larger fraction of the frame, which points to an advantage from stronger global context."
    )
if gain_summary["local_contrast"] < regression_summary["local_contrast"]:
    print(
        "Pattern signal: the gain group has weaker local contrast, so U-Net struggles more when the crack is visually closer to its surroundings."
    )
if gain_summary["ring_texture_std"] > regression_summary["ring_texture_std"]:
    print(
        "Pattern signal: the gain group sits on noisier nearby texture, which matches the gravel/background confusion seen in the deep-dive samples."
    )
if gain_summary["recall_delta"] > abs(gain_summary["precision_delta"]):
    print(
        "Pattern signal: most of SegFormer’s edge comes from recall, not precision, so the main U-Net issue is under-detection rather than over-segmentation."
    )
if gain_summary["unet_best_iou"] < 0.20:
    print(
        "Pattern signal: even after threshold sweep, U-Net remains weak on the gain group, so calibration alone is unlikely to close the gap."
    )

## 8. Actionable Hypotheses for Improving U-Net

The goal here is not to claim certainty from one notebook. Instead, we turn the observed pattern into a short list of concrete next experiments for the `U-Net` baseline.

In [ ]:
hypotheses = []

if gain_summary["bbox_fill_ratio"] < regression_summary["bbox_fill_ratio"]:
    hypotheses.append(
        {
            "title": "Increase effective resolution for thin cracks",
            "reason": (
                "Top-gain samples are thinner inside their GT bounding boxes, so U-Net is probably losing the crack before the decoder can recover it."
            ),
            "next_step": (
                "Try `img_size=512` or `640`, and pair it with foreground-aware crops so training batches contain more thin crack pixels."
            ),
        }
    )

if gain_summary["diag_span_ratio"] > regression_summary["diag_span_ratio"]:
    hypotheses.append(
        {
            "title": "Add more multi-scale or long-range context",
            "reason": (
                "Top-gain samples span more of the image, which suggests SegFormer benefits from seeing the crack as one long connected structure."
            ),
            "next_step": (
                "Test a larger crop, multi-scale training, or compare against `FPN` / `PSPNet` before making bigger architectural changes."
            ),
        }
    )

if gain_summary["local_contrast"] < regression_summary["local_contrast"] or gain_summary["ring_texture_std"] > regression_summary["ring_texture_std"]:
    hypotheses.append(
        {
            "title": "Train harder against low-contrast noisy backgrounds",
            "reason": (
                "The gain group is harder visually: weaker local contrast and/or rougher nearby texture. U-Net is likely over-relying on local edge strength."
            ),
            "next_step": (
                "Strengthen photometric augmentation, include more hard texture negatives, and consider contrast/shadow perturbation during validation stress tests."
            ),
        }
    )

if gain_summary["recall_delta"] > abs(gain_summary["precision_delta"]):
    hypotheses.append(
        {
            "title": "Bias the loss toward recall on sparse positives",
            "reason": (
                "SegFormer’s advantage is mainly recall, so U-Net is missing crack pixels more than it is hallucinating background."
            ),
            "next_step": (
                "Try `Tversky` or `Focal-Tversky`, or reweight the positive class inside `BCE + Dice` to punish false negatives more strongly."
            ),
        }
    )

if gain_summary["unet_best_iou"] < 0.20:
    hypotheses.append(
        {
            "title": "Do threshold tuning last, not first",
            "reason": (
                "On the gain group, even the best thresholded U-Net masks remain weak, so the logits themselves are not separating crack from background well enough."
            ),
            "next_step": (
                "Prioritize representation, sampling, and loss changes first; only re-tune the decision threshold after those experiments."
            ),
        }
    )

print("Recommended next U-Net experiments")
for i, item in enumerate(hypotheses, start=1):
    print(f"{i}. {item['title']}")
    print(f"   Why: {item['reason']}")
    print(f"   Try next: {item['next_step']}")
    print()

if not hypotheses:
    print("No strong automatic hypothesis fired. In that case, inspect more samples before deciding the next experiment.")